# 🐾 PetCare AI - Huấn Luyện ResNet50 Trên Google Colab (GPU Miễn Phí)
Notebook này huấn luyện model ResNet50 phân loại **4 bệnh ngoài da ở chó** từ bộ dữ liệu `Dogs.zip`.

### ⚡ Hướng dẫn:
1. Đảm bảo đã bật GPU: **Runtime** ➔ **Change runtime type** ➔ **T4 GPU** ➔ **Save**.
2. Chạy lần lượt các ô từ 1 đến 4.

In [ ]:
# 1. KIỂM TRA GPU
import torch
print('✅ Phiên bản PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print('🚀 Đang dùng GPU:', torch.cuda.get_device_name(0))
    print(f'⚡ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ CHÚ Ý: Chưa bật GPU! Vào Runtime > Change runtime type > Chọn T4 GPU')

In [ ]:
# 2 & 3. GIẢI NÉN VÀ TỰ ĐỘNG CHIA TRAIN (80%) / VALID (20%)
import os, shutil, zipfile, random
from google.colab import drive

# Mount Drive nếu có file Dogs.zip trong Drive
zip_path = None
if os.path.exists('/content/Dogs.zip'):
    zip_path = '/content/Dogs.zip'
elif os.path.exists('/content/drive/MyDrive/Dogs.zip'):
    zip_path = '/content/drive/MyDrive/Dogs.zip'
else:
    print('👉 Đang kết nối Google Drive để tìm Dogs.zip...')
    drive.mount('/content/drive')
    if os.path.exists('/content/drive/MyDrive/Dogs.zip'):
        zip_path = '/content/drive/MyDrive/Dogs.zip'

if not zip_path:
    raise FileNotFoundError('❌ Không tìm thấy file Dogs.zip! Hãy kéo file Dogs.zip vào mục Files ở cột bên trái hoặc vào Drive của bạn!')

print(f'📦 Đang giải nén từ: {zip_path}...')
extract_raw = '/content/raw_data'
if os.path.exists(extract_raw): shutil.rmtree(extract_raw)
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_raw)

# Tự động tìm thư mục chứa các lớp bệnh
target_classes = ['Bacterial_dermatosis', 'Fungal_infections', 'Healthy', 'Hypersensitivity_allergic_dermatosis']
found_dirs = {}
for root, dirs, _ in os.walk(extract_raw):
    for d in dirs:
        if d in target_classes:
            found_dirs[d] = os.path.join(root, d)

if len(found_dirs) < len(target_classes):
    # Fallback: lấy tất cả các folder con có ảnh
    for root, dirs, files in os.walk(extract_raw):
        imgs = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
        if len(imgs) > 10:
            folder_name = os.path.basename(root)
            found_dirs[folder_name] = root

print('✅ Các lớp bệnh tìm thấy:', list(found_dirs.keys()))
assert len(found_dirs) > 0, '❌ Không tìm thấy thư mục ảnh bệnh nào trong file zip!'

# Chuẩn bị thư mục /content/dataset/train và /content/dataset/valid
dataset_dir = '/content/dataset'
train_dir = os.path.join(dataset_dir, 'train')
valid_dir = os.path.join(dataset_dir, 'valid')
if os.path.exists(dataset_dir): shutil.rmtree(dataset_dir)
os.makedirs(train_dir, exist_ok=True)
os.makedirs(valid_dir, exist_ok=True)

random.seed(42)
valid_exts = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
classes = sorted(list(found_dirs.keys()))

print('\n--- TIẾN HÀNH CHIA TRAIN (80%) VÀ VALID (20%) ---')
for cls in classes:
    src = found_dirs[cls]
    tr_cls = os.path.join(train_dir, cls)
    va_cls = os.path.join(valid_dir, cls)
    os.makedirs(tr_cls, exist_ok=True)
    os.makedirs(va_cls, exist_ok=True)

    imgs = [f for f in os.listdir(src) if f.lower().endswith(valid_exts)]
    random.shuffle(imgs)
    split = int(len(imgs) * 0.8)
    for f in imgs[:split]: shutil.copy2(os.path.join(src, f), os.path.join(tr_cls, f))
    for f in imgs[split:]: shutil.copy2(os.path.join(src, f), os.path.join(va_cls, f))
    print(f'🔹 {cls}: Tổng {len(imgs)} ảnh -> Train: {split}, Valid: {len(imgs)-split}')

print('\n🎉 SẴN SÀNG HUẤN LUYỆN! Đã chia dữ liệu chuẩn vào /content/dataset/train và valid!')

In [ ]:
# 4. HUẤN LUYỆN MODEL RESNET50 TRÊN GPU (KHOẢNG 3 PHÚT)
import time
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from google.colab import files

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 15

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.RandomRotation(25),
        transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    'valid': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
}

train_ds = datasets.ImageFolder(train_dir, data_transforms['train'])
valid_ds = datasets.ImageFolder(valid_dir, data_transforms['valid'])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Khởi tạo ResNet50 Pretrained
print('>> Đang tải kiến trúc ResNet50 Pretrained...')
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

best_acc = 0.0
best_weights = '/content/disease_model.pth'

print(f'\n🚀 BẮT ĐẦU HUẤN LUYỆN {EPOCHS} EPOCHS TRÊN GPU...')
start_time = time.time()

for epoch in range(EPOCHS):
    # Train
    model.train()
    tr_loss, tr_corr = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * x.size(0)
        tr_corr += torch.sum(torch.max(out, 1)[1] == y.data).item()
    
    # Valid
    model.eval()
    va_loss, va_corr = 0.0, 0
    with torch.no_grad():
        for x, y in valid_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            va_loss += loss.item() * x.size(0)
            va_corr += torch.sum(torch.max(out, 1)[1] == y.data).item()
    
    tr_acc = tr_corr / len(train_ds)
    va_acc = va_corr / len(valid_ds)
    scheduler.step(va_loss / len(valid_ds))
    
    print(f'Epoch {epoch+1:02d}/{EPOCHS} | Train Acc: {tr_acc*100:.2f}% | Val Acc: {va_acc*100:.2f}%')
    if va_acc > best_acc:
        best_acc = va_acc
        torch.save(model.state_dict(), best_weights)
        print(f'   🌟 Lưu model kỷ lục mới: {best_acc*100:.2f}%')

print(f'\n🎉 HUẤN LUYỆN HOÀN TẤT trong {(time.time()-start_time)/60:.1f} phút!')
print(f'🌟 Độ chính xác cao nhất (Val Acc): {best_acc*100:.2f}%')

# Tự động tải model về máy tính
print('📥 Đang tải file disease_model.pth về máy tính của bạn...')
files.download(best_weights)